# 02 — 3000 картинок из Community Forensics + слияние с own/ (2000)

Датасет Community Forensics лежит на Hugging Face в parquet-файлах по 1–13 ГБ, каждый файл — один цельный блок,
поэтому обычная загрузка через `datasets` не помещается в память Colab. Здесь другой подход:

1. берём список файлов репозитория и оставляем только файлы **меньше `MAX_FILE_MB`**;
2. скачиваем их на диск по одному и читаем **только метаданные** (метка, генератор, промпт, источник real) — колонка
   с картинками при этом не читается, памяти нужно несколько десятков МБ;
3. по метаданным отбираем **1500 real + 1500 fake** из `N_GENERATORS` генераторов поровну, `N_HELD_OUT` из них —
   только для теста на невиденные модели;
4. достаём байты отобранных картинок, приводим к единому формату (JPEG, 512 по длинной стороне), пишем `metadata.csv`;
5. делаем split-файлы и вливаем ваши 2000 из `own.zip` (ноутбук 01).

GPU не нужен. Диска нужно ~20–30 ГБ (стандартный Colab даёт больше). Запускать сверху вниз.
Итог: папка `dataset/` → `images/` (5000 файлов), `metadata.csv`, `splits/`.

In [ ]:
!pip -q install -U huggingface_hub pyarrow pandas pillow tqdm

In [ ]:
import os, io, random, zipfile, shutil, collections
import pandas as pd
import pyarrow.parquet as pq
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from huggingface_hub import HfApi, hf_hub_download

CFG = dict(
    REPO = "OwensLab/CommunityForensics-Small",   # версия с real-картинками; основной "OwensLab/CommunityForensics" содержит только fake
    MAX_FILE_MB = 1500,       # файлы крупнее не берём — их не прочитать в памяти Colab
    MAX_TOTAL_GB = 25,        # суммарный лимит скачивания (диск Colab)
    N_REAL = 1500,
    N_FAKE = 1500,
    N_GENERATORS = 12,        # сколько генераторов в fake-части
    N_HELD_OUT = 3,           # сколько из них целиком уходят в test_unseen
    EXCLUDE_REAL_SOURCES = ["COCO"],   # ваши real в own/ — из COCO; если в CF хватает real из других источников, COCO не берём
    MAX_SIDE = 512, JPEG_Q = 95, SEED = 42,
    OUT = "/content/dataset",
    CACHE = "/content/hf_cache",
    OWN_ZIP = "/content/drive/MyDrive/deepfake_paper/own.zip",   # архив из ноутбука 01 (или положите в /content/own.zip)
)
random.seed(CFG["SEED"])
IMG_DIR = os.path.join(CFG["OUT"], "images"); SPL_DIR = os.path.join(CFG["OUT"], "splits")
for d in (IMG_DIR, SPL_DIR, CFG["CACHE"]):
    os.makedirs(d, exist_ok=True)

COLUMNS = ["id", "filename", "label", "origin", "source", "fake_type", "generator",
           "prompt", "caption", "original_id", "category", "source_url", "held_out", "split"]
META_COLS = ["image_name", "model_name", "label", "prompt", "real_source", "subset", "split", "format"]

## 0. Доступ к Hugging Face
Если дальше будет ошибка про gated/401: откройте страницу датасета на HF, нажмите «Agree», потом выполните эту ячейку с токеном.
Токен также снимает лимиты на скорость скачивания — лучше выполнить в любом случае.

In [ ]:
# from huggingface_hub import login
# login()

## 1. Список файлов репозитория
Берём parquet-файлы не крупнее `MAX_FILE_MB`, маленькие — первыми (в них обычно больше разных генераторов на мегабайт).

In [ ]:
info = HfApi().dataset_info(CFG["REPO"], files_metadata=True)
files = pd.DataFrame([(s.rfilename, (s.size or 0) / 1e6) for s in info.siblings], columns=["path", "mb"])
pqf = files[files.path.str.endswith(".parquet")].copy()
pqf["dir"] = pqf.path.str.rsplit("/", n=1).str[0]
print(f"всего файлов: {len(files)}, parquet: {len(pqf)}, объём parquet: {pqf.mb.sum()/1e3:.0f} ГБ")
print("\nпо папкам (кол-во файлов / ГБ):")
print(pqf.groupby("dir").agg(n=("path", "size"), gb=("mb", lambda s: round(s.sum()/1e3, 1))))

cand = pqf[pqf.mb <= CFG["MAX_FILE_MB"]].sort_values("mb").reset_index(drop=True)
print(f"\nкандидатов ≤ {CFG['MAX_FILE_MB']} МБ: {len(cand)} файлов, {cand.mb.sum()/1e3:.1f} ГБ")
print(cand.head(20).to_string())

## 2. Скачиваем файлы и читаем из них только метаданные
Останавливаемся, когда набрано достаточно: ≥ `N_REAL` real и ≥ `N_GENERATORS` генераторов с нужным числом картинок,
либо когда исчерпан `MAX_TOTAL_GB`. Если после остановки чего-то не хватает — увеличьте `MAX_TOTAL_GB` и перезапустите ячейку.

In [ ]:
per_gen = CFG["N_FAKE"] // CFG["N_GENERATORS"]
cat_parts, downloaded, total_mb = [], {}, 0.0

def enough(df):
    if df.empty:
        return False
    fake = df[df.label == 1]
    real = df[df.label == 0]
    real_ok = real[~real.real_source.isin(CFG["EXCLUDE_REAL_SOURCES"])] if len(real) else real
    gens_ok = (fake.model_name.value_counts() >= per_gen).sum()
    return len(real_ok) >= CFG["N_REAL"] and gens_ok >= CFG["N_GENERATORS"]

# чередуем папки, чтобы не набрать все файлы из одной (например, только Manual)
order = []
for i in range(cand.groupby("dir").size().max()):
    for d, grp in cand.groupby("dir"):
        if i < len(grp):
            order.append(grp.iloc[i])

for row in order:
    cat = pd.concat(cat_parts, ignore_index=True) if cat_parts else pd.DataFrame()
    if enough(cat):
        break
    if total_mb + row.mb > CFG["MAX_TOTAL_GB"] * 1e3:
        print("достигнут MAX_TOTAL_GB"); break
    local = hf_hub_download(CFG["REPO"], row.path, repo_type="dataset", cache_dir=CFG["CACHE"])
    total_mb += row.mb; downloaded[row.path] = local
    pf = pq.ParquetFile(local)
    cols = [c for c in META_COLS if c in pf.schema_arrow.names]
    t = pf.read(columns=cols).to_pandas()             # только метаданные — колонка image_data не читается
    t["label"] = pd.to_numeric(t["label"], errors="coerce").fillna(1).astype(int)
    for c in ("model_name", "prompt", "real_source", "subset"):
        if c not in t: t[c] = ""
    t["file"] = row.path; t["row"] = range(len(t))
    cat_parts.append(t)
    fk, rl = (t.label == 1).sum(), (t.label == 0).sum()
    print(f"{row.path}: {len(t)} строк, fake={fk} ({t[t.label==1].model_name.nunique()} моделей), real={rl}  | скачано {total_mb/1e3:.1f} ГБ")

cat = pd.concat(cat_parts, ignore_index=True)
cat["model_name"] = cat["model_name"].fillna("").astype(str)
cat["real_source"] = cat["real_source"].fillna("").astype(str)
print("\nИТОГО в каталоге:", len(cat), "| fake:", (cat.label == 1).sum(), "| real:", (cat.label == 0).sum())
print("\nreal по источникам:"); print(cat[cat.label == 0].real_source.value_counts())
print("\nfake по генераторам (топ-30 из", cat[cat.label == 1].model_name.nunique(), "):")
print(cat[cat.label == 1].model_name.value_counts().head(30))
cat.to_csv("/content/cf_catalog.csv", index=False)

## 3. Отбор 3000
Генераторы с ≥ `per_gen` картинок → случайно берём `N_GENERATORS`, из каждого по `per_gen`; `N_HELD_OUT` из них помечаем `held_out=1`.
Real: если хватает картинок из источников вне `EXCLUDE_REAL_SOURCES`, берём только их (чтобы ваши COCO-фото в own/ остались «внешними»).

In [ ]:
fake = cat[cat.label == 1]; real = cat[cat.label == 0]
counts = fake.model_name.value_counts()
eligible = counts[counts >= per_gen].index.tolist()
assert len(eligible) >= CFG["N_GENERATORS"], f"генераторов с ≥{per_gen} картинок всего {len(eligible)} — увеличьте MAX_TOTAL_GB или уменьшите N_GENERATORS"

chosen = random.sample(eligible, CFG["N_GENERATORS"])
held_out = set(random.sample(chosen, CFG["N_HELD_OUT"]))
print("выбранные генераторы:"); [print("  ", ("HELD-OUT " if g in held_out else "         "), g, counts[g]) for g in chosen]

real_pool = real[~real.real_source.isin(CFG["EXCLUDE_REAL_SOURCES"])]
if len(real_pool) < CFG["N_REAL"]:
    print(f"\nreal вне {CFG['EXCLUDE_REAL_SOURCES']} только {len(real_pool)} — беру все источники")
    real_pool = real
assert len(real_pool) >= CFG["N_REAL"], "real-картинок не хватает — увеличьте MAX_TOTAL_GB"

pick = [fake[fake.model_name == g].sample(per_gen, random_state=CFG["SEED"]) for g in chosen]
pick.append(real_pool.sample(CFG["N_REAL"], random_state=CFG["SEED"]))
sel = pd.concat(pick).sample(frac=1, random_state=CFG["SEED"]).reset_index(drop=True)
sel["held_out"] = ((sel.label == 1) & sel.model_name.isin(held_out)).astype(int)
print("\nотобрано:", len(sel), "| real:", (sel.label == 0).sum(), "| fake:", (sel.label == 1).sum(), "| held-out:", sel.held_out.sum())
print("real по источникам в выборке:"); print(sel[sel.label == 0].real_source.value_counts())

## 4. Достаём картинки и нормализуем
Для каждого файла читаем колонку `image_data` по одному блоку (row group) и берём из него только отобранные строки.
Прочитанный файл после этого удаляем с диска.

In [ ]:
def normalize(img):
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size; s = CFG["MAX_SIDE"] / max(w, h)
    return img.resize((round(w * s), round(h * s)), Image.LANCZOS) if s < 1 else img

sel["id"] = [f"cf_{i+1:06d}" for i in range(len(sel))]
sel["filename"] = sel["id"] + ".jpg"
done = set(os.listdir(IMG_DIR))

for fpath, grp in tqdm(sel.groupby("file"), desc="files"):
    need = {int(r.row): r for r in grp.itertuples()}
    if all(r.filename in done for r in need.values()):
        continue
    pf = pq.ParquetFile(downloaded[fpath])
    offset = 0
    for rg in range(pf.num_row_groups):
        n = pf.metadata.row_group(rg).num_rows
        rows_here = [r for r in need if offset <= r < offset + n]
        if rows_here:
            col = pf.read_row_group(rg, columns=["image_data"]).column("image_data")
            for r in rows_here:
                m = need[r]
                if m.filename in done:
                    continue
                img = Image.open(io.BytesIO(col[r - offset].as_py()))
                normalize(img).save(os.path.join(IMG_DIR, m.filename), "JPEG", quality=CFG["JPEG_Q"], optimize=True)
            del col
        offset += n
    try:
        os.remove(os.path.realpath(downloaded[fpath]))   # освобождаем диск
    except Exception:
        pass

cf = pd.DataFrame({
    "id": sel.id, "filename": sel.filename,
    "label": sel.label.map({0: "real", 1: "fake"}),
    "origin": "cf",
    "source": [m if l == 1 else (rs or "cf_real") for m, l, rs in zip(sel.model_name, sel.label, sel.real_source)],
    "fake_type": sel.label.map({0: "none", 1: "t2i"}),
    "generator": [m if l == 1 else "" for m, l in zip(sel.model_name, sel.label)],
    "prompt": sel.prompt.fillna(""), "caption": "", "original_id": "", "category": sel.subset.fillna(""),
    "source_url": "", "held_out": sel.held_out, "split": "",
}, columns=COLUMNS)
missing = [f for f in cf.filename if not os.path.exists(os.path.join(IMG_DIR, f))]
print("сохранено:", len(cf) - len(missing), "| не удалось:", len(missing))
cf = cf[~cf.filename.isin(missing)]
cf.to_csv(os.path.join(CFG["OUT"], "metadata_cf.csv"), index=False)

## 5. Split: train / val / test_seen / test_unseen
Held-out генераторы целиком → `test_unseen`. Остальное делится 70/15/15 внутри каждого генератора и внутри real.

In [ ]:
def assign_split(df, seed=CFG["SEED"]):
    df = df.copy(); df["split"] = ""
    df.loc[df.held_out == 1, "split"] = "test_unseen"
    rest = df[df.held_out != 1]
    for _, grp in rest.groupby(["label", "generator"]):
        idx = grp.sample(frac=1, random_state=seed).index.tolist()
        n = len(idx); n_tr = int(n * 0.70); n_va = int(n * 0.15)
        df.loc[idx[:n_tr], "split"] = "train"
        df.loc[idx[n_tr:n_tr + n_va], "split"] = "val"
        df.loc[idx[n_tr + n_va:], "split"] = "test_seen"
    return df

cf = assign_split(cf)
print(cf.groupby(["split", "label"]).size().unstack(fill_value=0))

## 6. Слияние с own/ (ноутбук 01) и финальные файлы

In [ ]:
own_zip = CFG["OWN_ZIP"]
if not os.path.exists(own_zip):
    if os.path.exists("/content/own.zip"):
        own_zip = "/content/own.zip"
    else:
        try:
            from google.colab import drive; drive.mount("/content/drive")
        except Exception:
            pass
own = pd.DataFrame(columns=COLUMNS)
if os.path.exists(own_zip):
    with zipfile.ZipFile(own_zip) as z:
        for n in tqdm([n for n in z.namelist() if n.startswith("images/") or n == "metadata.csv"], desc="unzip own"):
            z.extract(n, "/content/own_unpacked")           # originals/ не распаковываем — они не нужны
    own = pd.read_csv("/content/own_unpacked/metadata.csv", dtype=str).fillna("")
    own = own.reindex(columns=COLUMNS, fill_value="")
    own["split"] = "test_own"
    for fn in tqdm(own.filename, desc="copy own"):
        shutil.copy(f"/content/own_unpacked/images/{fn}", os.path.join(IMG_DIR, fn))
    print("own добавлено:", len(own))
else:
    print("own.zip не найден — пропускаю слияние (можно повторить эту ячейку позже)")

meta = pd.concat([cf, own], ignore_index=True)
meta.to_csv(os.path.join(CFG["OUT"], "metadata.csv"), index=False)
for s, grp in meta.groupby("split"):
    grp.to_csv(os.path.join(SPL_DIR, f"{s}.csv"), index=False)

assert meta.filename.is_unique and len(meta) == len(os.listdir(IMG_DIR)), "metadata и images/ разошлись"
print("\nИТОГО:", len(meta))
print(meta.groupby(["origin", "split", "label"]).size().unstack(fill_value=0))
print("\nТаблица для статьи (fake по генераторам):")
print(meta[meta.label == "fake"].groupby(["origin", "generator", "held_out"]).size())

## 7. Архив и сохранение в Google Drive

In [ ]:
shutil.make_archive("/content/dataset", "zip", root_dir=CFG["OUT"], base_dir=".")
try:
    from google.colab import drive; drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/deepfake_paper", exist_ok=True)
    shutil.copy("/content/dataset.zip", "/content/drive/MyDrive/deepfake_paper/dataset.zip")
    print("скопировано: Google Drive / deepfake_paper / dataset.zip")
except Exception as e:
    print("Drive не подключён, архив лежит в /content/dataset.zip", e)